# ReXKG Pipeline (PyHealth Style)

This notebook shows a PyHealth-native ReXKG workflow similar to other examples:

1. Load dataset
2. Set tasks
3. Build sample datasets
4. Create dataloaders
5. Run model forward smoke test

It also includes optional RUN_GUIDE shell steps for full KG construction.

In [ ]:
from pathlib import Path

from pyhealth.datasets import RexKGDataset, split_by_patient, get_dataloader
from pyhealth.tasks import (
    RexKGEntityExtractionRadiology,
    RexKGRelationExtractionRadiology,
    RexKGKnowledgeGraphConstruction,
)
from pyhealth.models import RexKG

## 1) Configure Paths

Set `PROJECT_ROOT` to your repo root if auto-detection does not match your environment.

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "PyHealth").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

REXKG_DATA_ROOT = PROJECT_ROOT / "src" / "ner" / "data"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("REXKG_DATA_ROOT:", REXKG_DATA_ROOT)
print("Data root exists:", REXKG_DATA_ROOT.exists())

## 2) Load RexKGDataset

In [ ]:
dataset = RexKGDataset(root=str(REXKG_DATA_ROOT))
dataset.stats()

## 3) Apply ReXKG Tasks

In [ ]:
entity_task = RexKGEntityExtractionRadiology()
relation_task = RexKGRelationExtractionRadiology()
kg_task = RexKGKnowledgeGraphConstruction()

entity_samples = dataset.set_task(entity_task)
relation_samples = dataset.set_task(relation_task)
kg_samples = dataset.set_task(kg_task)

print("Entity samples:", len(entity_samples))
print("Relation samples:", len(relation_samples))
print("KG samples:", len(kg_samples))

In [ ]:
if len(entity_samples) > 0:
    first = entity_samples[0]
    print("First entity sample keys:", sorted(first.keys()))
    print("Text chars:", len(first.get("text", "")))
else:
    print("No entity samples found. Check source CSV and text columns.")

## 4) Split + DataLoaders (PyHealth style)

Using entity task samples for downstream model/data checks.

In [ ]:
train_ds, val_ds, test_ds = split_by_patient(entity_samples, [0.7, 0.1, 0.2])

train_loader = get_dataloader(train_ds, batch_size=4, shuffle=True)
val_loader = get_dataloader(val_ds, batch_size=4, shuffle=False)
test_loader = get_dataloader(test_ds, batch_size=4, shuffle=False)

print("train/val/test sizes:", len(train_ds), len(val_ds), len(test_ds))

## 5) ReXKG Model Forward Smoke Test

This checks model wiring with a batch from the entity dataloader.

In [ ]:
import torch

model = RexKG(dataset=entity_samples, freeze_encoder=True)
batch = next(iter(train_loader))

if "text" not in batch:
    raise KeyError("Batch does not contain 'text'. Check task/input schema.")

with torch.no_grad():
    out = model(text=batch["text"])

print("Output keys:", sorted(out.keys()))
print("Entity logits shape:", tuple(out["entity_logits"].shape))
print("Relation logits shape:", tuple(out["relation_logits"].shape))

## 6) Optional: RUN_GUIDE Shell Pipeline

These commands are the original script-based ReXKG flow from `RUN_GUIDE.md`.
Run them from a terminal when you want full KG artifacts.

In [ ]:
run_guide_commands = [
    "cd src/ner/data && python structure_data.py",
    "cd src/ner && sh run_entity.sh",
    "cd src/ner && sh run_relation.sh",
    "cd src/ner && sh run_inference.sh",
    "cd src/ner/result/run_relation && python reverse_structure_data.py --input_json_file ../run_relation/predictions.json --save_json_file ../../data/your_test_file.json",
    "cd src/kg_construct/code && python get_entities.py --ent_pred_mimic_headct ../../ner/data/your_test_file.json --ent_real_pred_mimic_headct ../../ner/data/your_test_file.json --save_entity_dir ../result/your_run/entities --save_real_dir ../result/your_run/relation",
    "cd src/kg_construct/code && python get_umls_entities.py --save_entity_dir ../result/your_run/entities",
    "cd src/kg_construct/code && python filter_cui.py --save_entity_dir ../result/your_run/entities",
    "cd src/kg_construct/code && python structure_entities.py --save_entity_dir ../result/your_run/entities --ignore_count 10",
    "cd src/kg_construct/code && python get_kg_nodes.py --save_entity_dir ../result/your_run/entities --save_real_dir ../result/your_run/relation --save_kg_dir ../result/your_run/kg",
    "cd src/kg_construct/code && python get_size_relations.py --entity_dir ../result/your_run/entities --real_dir ../result/your_run/relation",
]

for c in run_guide_commands:
    print(c)